# Groepsgrootte 16 met vaste schaal ×64 / A32 — vijf-epochpilot

Deze proef vergelijkt **groepsgrootte 16 met de bestaande groepsgrootte 32**, beide met vaste ×64 voor de lagere precisies. E4M3 wordt uit de actuele FP32-gewichten bepaald. E2M3 behoudt de hoogste twee exponentbits en gebruikt **E8M0-broncode +6**. E2M1 verwijdert daarna twee mantissabits van E2M3 en behoudt dezelfde E2M3-schaal. Eén E4M3-bron wordt per laag en minibatch gedeeld door de drie modi.

De groepering blijft per outputkanaal over de afgevlakte invoergewichten, met nulpadding aan het einde van elke rij. Activaties zijn **A32**; de gedeelde updatemaster, gradients en Adam-momenten zijn **FP32**. Er zijn geen geleerde schaalparameters. De bronschaal wordt opnieuw bepaald uit de actuele gewichten; de lagere offset blijft +6.

**De hoofdfiguren tonen epoch 5 en het gezamenlijke checkpoint gekozen door de hoogste E4M3-accuracy binnen deze pilot.** De B32-referentie wordt voor beide vergelijkingen tot dezelfde vroege epochs beperkt. Epoch 100 is voor de B16-proef nog niet gemeten.

De kostenanalyse toont apart het aantal schalen, de padding en de totale logische opslag voor B16, B32 en B64. **B64 is hier alleen een geometrische kostenberekening, geen gemeten accuracyproef.**

Dit notebook leest uitsluitend opgeslagen JSON/CSV en rekent geometrische opslagkosten na. Het laadt geen dataset, model, checkpoint of optimizer en start geen training.


In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # De runkopie bevat hier het exacte resultaatpad.
SAVE_FIGURES = False  # True schrijft naar een nieuwe analysis_<tijdstip>-map.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

# Gebruik de helper uit déze run, ook als een eerdere notebook dezelfde modulenaam gebruikte.
helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
helper_name = '_mx_triple_group16_scale64_pilot5_analysis_helper'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
if spec is None or spec.loader is None:
    raise FileNotFoundError(f'Analysehelper ontbreekt: {helper_path}')
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Epoch 5 en beste checkpoint gekozen door E4M3

Het beste checkpoint is het **eerste strikte maximum van de E4M3-testaccuracy binnen de pilot**. Bij gelijke scores blijft de eerdere epoch geselecteerd. E2M3 en E2M1 worden op dezelfde epoch gerapporteerd; hun eigen hoogste scores worden niet met een andere E4M3-epoch samengevoegd.

De officiële testset wordt volgens het bestaande protocol voor checkpointselectie gebruikt. Die beste score is geen onafhankelijke testschatting na selectie. Het vaste eindpunt epoch 5 staat er apart naast. Dit is een vroege trainingsproef; de volledige vergelijking op epoch 100 volgt pas na een afzonderlijk voltooide honderd-epochtraining.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'Harde test-CE': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


## Vergelijking met groepsgrootte 32: dezelfde ×64-correctie en dezelfde vroege epochs

De ongewijzigde B32/×64-referentie bevat de eerdere honderd-epochtraining. Hieronder worden **uitsluitend epochs 1 tot en met 5** gebruikt. De curves stoppen bij de laatste beschikbare B16-epoch. De epoch-5-vergelijking verschijnt pas wanneer B16 epoch 5 bereikt heeft.

Voor de beste-checkpointvergelijking selecteert E4M3 per training één epoch **binnen de gezamenlijke vroege epochrange**. Alle drie modi horen bij dat ene checkpoint. De globale beste epoch 42 en epoch 100 van de B32-referentie worden niet als pilotcheckpoint gebruikt. Referentiehashes, oorspronkelijke initialisatie en trainingsinstellingen worden gecontroleerd. De bedoelde experimentele wijziging is de groepsgrootte; de factor blijft ×64.


In [ ]:
reference = ns.load_scale64_reference(run)
display(Markdown(ns.comparison_markdown(run, reference)))
display(ns.comparison_table(run, reference).round(6))
figures['group_comparison_curves'] = ns.plot_scale_comparison_curves(run, reference)
display(figures['group_comparison_curves'])
plt.close(figures['group_comparison_curves'])
figures['group_comparison_checkpoints'] = ns.plot_scale_comparison_checkpoints(run, reference)
display(figures['group_comparison_checkpoints'])
plt.close(figures['group_comparison_checkpoints'])


## Kosten: aantal schalen, padding en totale opslag

Elke groep heeft één E8M0-schaal van acht bits. Zonder padding kost de schaal $8/B$ bits per gewicht: 0,5 voor B16, 0,25 voor B32 en 0,125 voor B64. Onze implementatie groepeert echter per outputrij en vult elke rij tot een veelvoud van B aan.

De berekening gebeurt per laag en per precisie. Voor E4M3 tonen we ook de afzonderlijke overhead van schaalbytes en paddingcodes bovenop de reële gewichtswaarden. Dit maakt zichtbaar waarom **B64 in dit netwerk minder schalen maar toch meer bronopslag dan B32 heeft**.


In [ ]:
display(Markdown(ns.cost_markdown(run)))
display(ns.cost_summary_table(run).round(6))
figures['group_size_costs'] = ns.plot_group_costs(run)
display(figures['group_size_costs'])
plt.close(figures['group_size_costs'])


### Kosten per laag en controle tegen de opgeslagen exports

De eerste tabel geeft de berekende E4M3-bronindeling per laag bij B16/B32/B64. De lagere precisies worden met dezelfde groepsgeometrie afzonderlijk berekend. `group_size_costs_layers.csv` bevat alle lagen en alle drie de precisies.

De tweede tabel controleert B16 tegen de opgeslagen exportdiagnostiek op het beste en laatste checkpoint. Deze controle bevestigt gerapporteerde aantallen; de codec- en exporttests van de runner controleren daarnaast de werkelijke codebytes en gereconstrueerde gewichten.


In [ ]:
display(ns.cost_layer_table(run, weight_bits=8))
display(ns.export_layout_table(run))


## Kwantisatiefout op de twee geselecteerde checkpoints

De opgeslagen exportdiagnostiek vergelijkt de gecodeerde gewichten met de effectieve FP32-gewichten G(W), zonder een nieuwe evaluatie te starten. RMSE wordt gewogen over de reële gewichten van de zes MX-lagen; padding telt niet mee. Nieuwe nulwaarden zijn gewichten die vóór kwantisatie niet nul waren en daarna nul worden. Dit is gewichtsdiagnostiek, geen vervanging voor testaccuracy.


In [ ]:
display(ns.quantization_table(run))
figures['quantization_diagnostics'] = ns.plot_quantization(run)
display(figures['quantization_diagnostics'])
plt.close(figures['quantization_diagnostics'])


## Train- en testaccuracy per modus

De drie modi worden gezamenlijk vanaf dezelfde oorspronkelijke initialisatie getraind. Trainaccuracy wordt tijdens de updates verzameld. Testaccuracy wordt na iedere epoch gemeten met B16 en dezelfde ×64-truncatieroute, bevroren evaluatiestaat en de volledige testset. De drie modusgebonden BatchNorm-banken leren tijdens de training normaal mee.


In [ ]:
figures['accuracy_curves'] = ns.plot_learning(run)
display(figures['accuracy_curves'])
plt.close(figures['accuracy_curves'])


## Losses: trainingsdoelen onderscheiden van harde test-CE

De trainpass van E4M3 gebruikt harde CE van labels. Daarna leert E2M3 met soft-target CE van de **detached E4M3-softmax**; E2M1 leert met soft-target CE van de **detached E2M3-softmax**. Dit is recursieve distillatie met T=1. De drie gradients worden gesommeerd zonder deling door drie, waarna één Adam-update volgt; er is geen extra FP32-teacherforward.

De drie trainobjectieven staan in afzonderlijke panelen omdat hun targets verschillen. Soft CE is geen harde label-CE en wordt hier ook niet als KL-divergentie aangeduid. **Testloss is voor alle drie modi harde CE van dezelfde labels** en staat daarom samen in één vergelijkbaar paneel.


In [ ]:
if not run.metrics.empty:
    objective_columns = [key for key in ('epoch', 'phase', 'weight_format', 'objective_type', 'objective_loss') if key in run.metrics]
    display(run.metrics[objective_columns].round({'objective_loss': 8}))
figures['objective_losses'] = ns.plot_objectives(run)
display(figures['objective_losses'])
plt.close(figures['objective_losses'])


## Blijft het netwerk verschillende cijfers voorspellen?

De eerdere post-hocproeven lieten bij sommige routes sterk vertekende of constante voorspellingen zien. Daarom wordt hier per testepoch en per modus bijgehouden hoe de voorspellingen over de tien cijfers verdeeld zijn. Een constante voorspelling is direct zichtbaar als **één gebruikte klasse en 100% concentratie**. De tabel toont ook hoe vaak het dominante cijfer werkelijk in de testset voorkomt.

Dit zijn geaggregeerde tellingen; deze notebook doet geen nieuwe inferentie en leidt er geen per-beeldvergelijking uit af.


In [ ]:
display(ns.prediction_table(run, selected_only=True))
figures['prediction_diversity'] = ns.plot_prediction_diversity(run)
display(figures['prediction_diversity'])
plt.close(figures['prediction_diversity'])


In [ ]:
display(Markdown('## Gemeten trainingsduur en voorlopige projectie naar honderd epochs'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nDe opgeslagen pieken betreffen de PyTorch CUDA-allocator. Zij meten niet alle GPU-processen of compacte native MX-uitvoering. Ontbrekende waarden blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabellen rapporteren bestaande controles uit de run. Het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates\n\nDe mastergewichten zijn dezelfde gedeelde tensors. De herhaalde masterkolommen per modus mogen **niet bij elkaar worden opgeteld**; code- en schaalwijzigingen behoren wel bij het betreffende formaat.'))
display(ns.weight_update_table(run))
display(Markdown('### Eén opgeslagen E4M3-bron en twee afgeleide exports per geselecteerd checkpoint'))
display(ns.export_table(run))


## Interpretatie en grenzen

De gerichte wijziging tegenover de bestaande B32/×64-run is **groepsgrootte 16 in plaats van 32**. De factor blijft ×64, de mantissatruncatie behoudt de E2M3-schaal, en de overige trainingsinstellingen en initiële mastergewichten komen overeen. Er is geen directe lagere kwantisatie, geleerde schaal of extra FP32-teacherpass. De identity-STE geldt voor de volledige codebewerking naar G(W), zonder extra factor64 in de backward.

Een lage accuracy blijft een onderzoeksuitkomst wanneer de technische controles slagen. Een dalende soft-target trainingsloss bewijst op zichzelf geen betere classificatie; bekijk testaccuracy, harde test-CE en voorspellingstellingen samen. Kleinere groepen kunnen een gunstigere lokale schaal opleveren, maar garanderen geen hogere netwerkaccuracy.

De oorspronkelijke cuDNN-backend garandeert geen bitexacte CUDA-herhaalbaarheid. Deze proef heeft één seed en vijf epochs; een klein verschil bewijst geen algemene superioriteit of optimale groepsgrootte. De beste E4M3-score selecteert op de officiële testset en is geen onafhankelijke testschatting.

**Kosten horen bij de conclusie:** meer groepen betekenen meer schaalbytes, terwijl grotere groepen bij onze rijindeling meer padding kunnen veroorzaken. Rapporteer daarom zowel schaaloverhead als totale payload. De drie exportgroottes zijn alternatieven; één E4M3-bronmodel wordt niet driemaal geteld. Logische payload, checkpointbestandsgrootte en gemeten GPU-geheugen zijn verschillende grootheden.

De tijdprojectie voor honderd epochs is uitsluitend een looptijdschatting. Er wordt geen accuracy voor epoch 100 berekend. De geheugenfiguur betreft de CUDA-allocator van de FP32-emulator en meet geen native MX-kernel, hardwareversnelling of energiewinst.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')
print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [str(path.relative_to(run.path)) if path.is_relative_to(run.path) else str(path) for path in run.input_files]}))
